In [1]:
from  pydantic import BaseModel, Field
import instructor
import openai
from qdrant_client import QdrantClient

/home/nurda/ai-engineering/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_API_KEY = "ollama"

CHAT_MODEL = "qwen2.5:7b"
EMBED_MODEL = "nomic-embed-text"

clientOllama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=OLLAMA_API_KEY,
)

### Add instructor

In [3]:
prompt = """You are a helpful shopping assistant that can answer questions about products based on the available products."""

In [15]:
client = instructor.from_openai(clientOllama, mode=instructor.Mode.JSON,)

In [5]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question.")

In [26]:
response, raw_response = client.chat.completions.create_with_completion(
        model=CHAT_MODEL,
        response_model=RAGGenerationResponse,
        messages=[{"role": "system", "content": prompt}],
        temperature=0.5,
    )

In [28]:
print(response.answer)

I can certainly help with that! Could you please provide me with more details about the products you're interested in or your specific question? That way, I'll be able to give you an accurate answer.


In [29]:
print(raw_response)

ChatCompletion(id='chatcmpl-927', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "answer": "I can certainly help with that! Could you please provide me with more details about the products you\'re interested in or your specific question? That way, I\'ll be able to give you an accurate answer."\n}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1776670597, model='qwen2.5:7b', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=49, prompt_tokens=123, total_tokens=172, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


### RAG pipeline


In [47]:
def get_embedding(text, model="nomic-embed-text"):
    response = client.embeddings.create(
        input=text,
        model=model,
    )

    return response.data[0].embedding


def retrieve_data(query , qdrant_client, top_k=5):
    query_embedding = get_embedding(query)
    search_result = qdrant_client.query_points(
        collection_name="Amazon_items_collection1",
        query=query_embedding,
        limit=top_k
    )

    retrieved_context_ids = []
    retrieved_contexts = []
    similarity_scores = []
    retrieved_context_ratings = []

    for point in search_result.points:
        retrieved_context_ids.append(point.payload["parent_asin"])
        retrieved_contexts.append(point.payload["description"])
        retrieved_context_ratings.append(point.payload["average_rating"])
        similarity_scores.append(point.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_contexts": retrieved_contexts,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):
    processed_context = ""
    for id_, chunk, rating in zip(
        context["retrieved_context_ids"],
        context["retrieved_contexts"],
        context["retrieved_context_ratings"]
    ):
        processed_context += f"-ID {id_}: rating: {rating}, description: {chunk}\n"
    return processed_context

def build_prompt(query, context):
    prompt = f"""You are shopping assistant that can answer questions about products. 
    You will be given a question and some retrieved information about products.
    
    Instructions:
    - You need to use only the retrieved information to answer the question.
    - Never use word context and refer to it as the available products.

    Context:
    {context}

    Question:
    {query}  """

    return prompt


def generate_answer(prompt):
    
    client = instructor.from_provider(
        "ollama/qwen2.5:7b",
            base_url="http://localhost:11434/v1",
            api_key="ollama",
        mode = instructor.Mode.JSON,
    )
    response, raw_response = client.chat.completions.create_with_completion(
        model=CHAT_MODEL,
        response_model=RAGGenerationResponse,
        messages=[{"role": "system", "content": prompt}],
        temperature=0.5,
    )

    return response


def rag_pipeline(query, qdrant_client, top_k=5):
    retrieved_data = retrieve_data(query, qdrant_client, top_k)
    processed_context = process_context(retrieved_data)
    prompt = build_prompt(query, processed_context)
    answer = generate_answer(prompt)

    final_result = {
        "datamodel": answer,
        "answer": answer.answer,
        "question": query,
        "retrieved_context": retrieved_data["retrieved_contexts"],
        "retrieved_context_ids": retrieved_data["retrieved_context_ids"],
        "similarity_scores": retrieved_data["similarity_scores"],
    }

    return final_result

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

/home/nurda/ai-engineering/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:288: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


In [48]:
output = rag_pipeline("What are the best wireless headphones?", qdrant_client)

In [49]:
print(output)

{'datamodel': RAGGenerationResponse(answer='The best wireless headphones among the available options are ID B0BK8WW5DH and ID B0BTBHYWY4. ID B0BK8WW5DH offers advanced noise cancellation, long battery life, and high-quality sound with Bluetooth 5.2 technology. ID B0BTBHYWY4 provides excellent comfort and noise isolation for over-ear headphones, making them perfect for extended listening sessions.'), 'answer': 'The best wireless headphones among the available options are ID B0BK8WW5DH and ID B0BTBHYWY4. ID B0BK8WW5DH offers advanced noise cancellation, long battery life, and high-quality sound with Bluetooth 5.2 technology. ID B0BTBHYWY4 provides excellent comfort and noise isolation for over-ear headphones, making them perfect for extended listening sessions.', 'question': 'What are the best wireless headphones?', 'retrieved_context': ['Earbuds Wireless Bluetooth,Bluetooth 5.2 Earbuds,CVC8.0 Noise Cancelling Earbuds, IPX7 Waterproof Stereo in ear Earphones with LED Power Display,Deep B

In [50]:
print(output["datamodel"])

answer='The best wireless headphones among the available options are ID B0BK8WW5DH and ID B0BTBHYWY4. ID B0BK8WW5DH offers advanced noise cancellation, long battery life, and high-quality sound with Bluetooth 5.2 technology. ID B0BTBHYWY4 provides excellent comfort and noise isolation for over-ear headphones, making them perfect for extended listening sessions.'


In [43]:
output["answer"]

'The best wireless headphones among the given options could be the zivsivc Bluetooth Earbuds with ID B0BK8WW5DH due to their high-quality sound, long battery life, advanced noise cancellation, and water resistance features.'

### Grounding context

In [51]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="The ID of the retrieved context.")
    description: str = Field(description="The description of the retrieved context.")

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question.")
    references : list[RAGUsedContext] = Field(description="List of retrieved contexts from the vector database.")

In [52]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- As an output you need to provide:

* The answer to the question based on the provided context.
* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
* Short description (1-2 sentences) of the item based on the description provided in the context.

- The short description should have the name of the item.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt

In [53]:
def get_embedding(text, model="nomic-embed-text"):
    response = client.embeddings.create(
        input=text,
        model=model,
    )

    return response.data[0].embedding


def retrieve_data(query , qdrant_client, top_k=5):
    query_embedding = get_embedding(query)
    search_result = qdrant_client.query_points(
        collection_name="Amazon_items_collection1",
        query=query_embedding,
        limit=top_k
    )

    retrieved_context_ids = []
    retrieved_contexts = []
    similarity_scores = []
    retrieved_context_ratings = []

    for point in search_result.points:
        retrieved_context_ids.append(point.payload["parent_asin"])
        retrieved_contexts.append(point.payload["description"])
        retrieved_context_ratings.append(point.payload["average_rating"])
        similarity_scores.append(point.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_contexts": retrieved_contexts,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):
    processed_context = ""
    for id_, chunk, rating in zip(
        context["retrieved_context_ids"],
        context["retrieved_contexts"],
        context["retrieved_context_ratings"]
    ):
        processed_context += f"-ID {id_}: rating: {rating}, description: {chunk}\n"
    return processed_context

def generate_answer(prompt):
    
    client = instructor.from_provider(
        "ollama/qwen2.5:7b",
            base_url="http://localhost:11434/v1",
            api_key="ollama",
        mode = instructor.Mode.JSON,
    )
    response, raw_response = client.chat.completions.create_with_completion(
        model=CHAT_MODEL,
        response_model=RAGGenerationResponse,
        messages=[{"role": "system", "content": prompt}],
        temperature=0.5,
    )

    return response


def rag_pipeline(query, qdrant_client, top_k=5):
    retrieved_data = retrieve_data(query, qdrant_client, top_k)
    processed_context = process_context(retrieved_data)
    prompt = build_prompt(query, processed_context)
    answer = generate_answer(prompt)

    final_result = {
        "output": answer,
        "answer": answer.answer,
        "question": query,
        "references": answer.references,
        "retrieved_context": retrieved_data["retrieved_contexts"],
        "retrieved_context_ids": retrieved_data["retrieved_context_ids"],
        "similarity_scores": retrieved_data["similarity_scores"],
    }

    return final_result

In [54]:
output = rag_pipeline("Can i get chargers for iphone?", qdrant_client)

In [55]:
output

{'output': RAGGenerationResponse(answer='- Light Up iPhone Charger Cable - MFi Certified Apple RGB Light LED Charging Cord: USB A to Lightning Cable for iPhone, iPad, iPod and More (5ft)\n- Features:\n  - Alternating blue, red, green, purple, yellow, pink lighting\n  - Ultra-fast charging with data transfer speeds up to 480Mb/s & fast charging speeds of up to 3 amps\n  - Compatible with iPhone 14, 13, 12, 11, X, 8, 7, 6, 5 series models and iPad, iPod Touch\n  - High-quality materials: metal shell for grip and soft strong PVC wire for wear-resistance\n  - Bending life exceeds 10,000 times to reduce frequent replacement\n- What you will get:\n  - 1 piece of 5-foot light up iPhone charger cable\n  - 12-month product warranty with prompt customer service response within 24 hours', references=[RAGUsedContext(id='B0BNQ951N8', description='Light Up iPhone Charger Cable - MFi Certified Apple RGB Light LED Charging Cord.USB A to Lightning Cable for iPhone,iPad,iPod and More (5ft)')]),
 'answer

In [56]:
print(output["answer"])

- Light Up iPhone Charger Cable - MFi Certified Apple RGB Light LED Charging Cord: USB A to Lightning Cable for iPhone, iPad, iPod and More (5ft)
- Features:
  - Alternating blue, red, green, purple, yellow, pink lighting
  - Ultra-fast charging with data transfer speeds up to 480Mb/s & fast charging speeds of up to 3 amps
  - Compatible with iPhone 14, 13, 12, 11, X, 8, 7, 6, 5 series models and iPad, iPod Touch
  - High-quality materials: metal shell for grip and soft strong PVC wire for wear-resistance
  - Bending life exceeds 10,000 times to reduce frequent replacement
- What you will get:
  - 1 piece of 5-foot light up iPhone charger cable
  - 12-month product warranty with prompt customer service response within 24 hours
